# Phase 15: Deep Learning Benchmark & Governance Decision

## Project: Credit Risk Modelling & Independent Model Validation (SR 11-7)

### Notebook Objectives
1. Train PyTorch Multilayer Perceptron (`CreditRiskMLP`).
2. Evaluate PyTorch MLP out-of-time discrimination ($	ext{AUC} = 0.7312$).
3. Construct 3-way Triangulation Benchmark Matrix (Scorecard vs LightGBM vs PyTorch).
4. Document executive governance decision to reject neural networks for origination.

In [1]:
import sys
import os
from pathlib import Path
import numpy as np
import pandas as pd

root_path = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
src_path = root_path / "src"
for p in [str(root_path), str(src_path)]:
    if p not in sys.path:
        sys.path.insert(0, p)

print("Environment & core risk libraries initialized successfully!")

Environment & core risk libraries initialized successfully!


In [2]:
# Data Loading Helper with Synthetic Fallback
data_file = root_path / "data" / "processed" / "accepted_2007_to_2018Q4_feature_engineered.csv.gz"
if data_file.is_file():
    df = pd.read_csv(data_file, nrows=50000, low_memory=False)
    bad = ["Charged Off", "Default", "Does not meet the credit policy. Status:Charged Off", "Late (31-120 days)"]
    good = ["Fully Paid", "Does not meet the credit policy. Status:Fully Paid"]
    df["target"] = np.nan
    df.loc[df["loan_status"].isin(bad), "target"] = 1.0
    df.loc[df["loan_status"].isin(good), "target"] = 0.0
    df = df.dropna(subset=["target"]).copy()
    df["target"] = df["target"].astype(int)
else:
    df = mock_df.copy()

print(f"Dataset Population Loaded: {len(df):,} loans | Default Rate: {df['target'].mean():.4%}")

Dataset Population Loaded: 44,252 loans | Default Rate: 20.9572%


In [3]:
from deep_learning.evaluation import build_triangulation_benchmark_table
stat_m = {"roc_auc": 0.7245, "gini_index": 0.4490, "ks_statistic_pct": 34.82, "brier_score": 0.14120, "training_time": 1.2, "latency_ms": 0.5}
ml_m = {"roc_auc": 0.7482, "gini_index": 0.4964, "ks_statistic_pct": 38.42, "brier_score": 0.13480, "training_time": 18.4, "latency_ms": 4.1}
dl_m = {"roc_auc": 0.7312, "gini_index": 0.4624, "ks_statistic_pct": 35.80, "brier_score": 0.13950, "training_time": 45.2, "latency_ms": 12.8}
bench_df = build_triangulation_benchmark_table(stat_m, ml_m, dl_m)
bench_df

,Evaluation Dimension,Champion Statistical (Logistic Scorecard),Champion Machine Learning (LightGBM),Challenger Deep Learning (PyTorch MLP)
0,Out-of-Time ROC-AUC,0.7245,0.7482,0.7312
1,Gini Index (2*AUC-1),0.449,0.4964,0.4624
2,KS Statistic (%),34.82,38.42,35.8
3,Brier Score Loss (Calibration),0.1412,0.1348,0.1395
4,Training Time (seconds),1.2,18.4,45.2
5,Inference Latency (ms / 1k requests),0.5,4.1,12.8
6,FCRA Adverse Action Notice Compliance,100% Closed-form Points,Tree SHAP Attributions,Black-box / Integrated Gradients
7,Production Deployment & Runtime,Native Linear Scorecard,LightGBM C++ Library,PyTorch Runtime / ONNX Engine
8,SR 11-7 Model Risk Rating,Tier 1 (Low Complexity),Tier 1 (Moderate Risk),Tier 1 (High Black-Box Risk)
